# Healthcare Spending Survey: Data Analysis

This notebook loads the survey responses exported from MongoDB (`data/user_data.csv`, produced by `export_to_csv.py`) and creates the charts for the client presentation:

1. **The ages with the highest income**
2. **The gender distribution across spending categories**

All charts are saved as high-resolution PNG files in the `charts/` folder so they can be inserted into PowerPoint.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

DATA_PATH = Path("data/user_data.csv")
CHARTS_DIR = Path("charts")
CHARTS_DIR.mkdir(exist_ok=True)

CATEGORIES = {
    "utilities": "Utilities",
    "entertainment": "Entertainment",
    "school_fees": "School Fees",
    "shopping": "Shopping",
    "healthcare": "Healthcare",
}


def save_chart(fig, name):
    """Save a chart as a 300-dpi PNG for PowerPoint."""
    path = CHARTS_DIR / name
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved {path}")

## 1. Load and inspect the data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"{len(df)} responses, {df.shape[1]} columns")
df.head()

In [ ]:
df.info()
df.describe().round(2)

## 2. Clean the data

In [ ]:
# Drop incomplete rows and make sure numeric columns are numeric.
numeric_cols = ["age", "total_income", *CATEGORIES, "total_expenses"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
df = df.dropna(subset=["age", "gender", "total_income"])
df[list(CATEGORIES)] = df[list(CATEGORIES)].fillna(0)
df["age"] = df["age"].astype(int)

# Age bands make the trend easier to read for a client audience.
bins = [17, 24, 34, 44, 54, 64, 120]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels)

print(f"{len(df)} rows after cleaning")
df.gender.value_counts()

## 3. Ages with the highest income

First, the 10 individual ages with the highest **average** total income; then average income by age group to show the overall trend.

In [ ]:
income_by_age = (
    df.groupby("age")["total_income"]
      .agg(average_income="mean", respondents="count")
      .sort_values("average_income", ascending=False)
)
top_ages = income_by_age.head(10)
top_ages.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_data = top_ages.reset_index().sort_values("average_income")
bars = ax.barh(plot_data["age"].astype(str), plot_data["average_income"], color="#0b6e75")
ax.bar_label(bars, labels=[f"${v:,.0f}" for v in plot_data["average_income"]], padding=4)
ax.set_title("Top 10 Ages by Average Total Income", fontsize=15, weight="bold")
ax.set_xlabel("Average Total Income ($)")
ax.set_ylabel("Age")
ax.xaxis.set_major_formatter(lambda x, _: f"${x:,.0f}")
ax.margins(x=0.12)
save_chart(fig, "01_top_ages_by_income.png")
plt.show()

In [ ]:
income_by_group = df.groupby("age_group", observed=True)["total_income"].mean()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(income_by_group.index.astype(str), income_by_group.values, color="#3a9ca3")
ax.bar_label(bars, labels=[f"${v:,.0f}" for v in income_by_group.values], padding=3)
ax.set_title("Average Total Income by Age Group", fontsize=15, weight="bold")
ax.set_xlabel("Age Group")
ax.set_ylabel("Average Total Income ($)")
ax.yaxis.set_major_formatter(lambda y, _: f"${y:,.0f}")
save_chart(fig, "02_income_by_age_group.png")
plt.show()

## 4. Gender distribution across spending categories

We reshape the expense columns into long format, then look at:
- **total amount spent** in each category by gender, and
- **number of participants** who reported spending in each category, by gender.

In [ ]:
spending = df.melt(
    id_vars=["gender"],
    value_vars=list(CATEGORIES),
    var_name="category",
    value_name="amount",
)
spending["category"] = spending["category"].map(CATEGORIES)

total_by_gender = spending.pivot_table(index="category", columns="gender", values="amount", aggfunc="sum")
total_by_gender.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=spending, x="category", y="amount", hue="gender", estimator="sum", errorbar=None, ax=ax)
ax.set_title("Total Spending by Category and Gender", fontsize=15, weight="bold")
ax.set_xlabel("Spending Category")
ax.set_ylabel("Total Amount Spent ($)")
ax.yaxis.set_major_formatter(lambda y, _: f"${y:,.0f}")
ax.legend(title="Gender", bbox_to_anchor=(1.02, 1), loc="upper left")
save_chart(fig, "03_spending_by_category_and_gender.png")
plt.show()

In [ ]:
spenders = spending[spending["amount"] > 0]
counts = spenders.groupby(["category", "gender"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(11, 6))
counts.plot(kind="bar", stacked=True, ax=ax, colormap="tab10", width=0.7)
ax.set_title("Number of Participants Spending in Each Category, by Gender", fontsize=15, weight="bold")
ax.set_xlabel("Spending Category")
ax.set_ylabel("Number of Participants")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Gender", bbox_to_anchor=(1.02, 1), loc="upper left")
save_chart(fig, "04_participants_by_category_and_gender.png")
plt.show()
counts

## 5. Summary

The charts in the `charts/` folder are ready to insert into PowerPoint (**Insert → Pictures → This Device**):

| File | Shows |
|---|---|
| `01_top_ages_by_income.png` | The 10 ages with the highest average income |
| `02_income_by_age_group.png` | Average income trend across age groups |
| `03_spending_by_category_and_gender.png` | Total spending in each category, split by gender |
| `04_participants_by_category_and_gender.png` | How many people of each gender spend in each category |